# 🗂️ Notebook 2: Gmail — Data Model & APIs

Now that we've sketched the architecture, let's get concrete: what do the tables,
objects, and HTTP endpoints look like? We'll build each piece in Python so you
can run, inspect, and break it.

## 🛠️ Setup

```bash
cd 06-system-designs/gmail
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

## 1. The data model

```
users(id, email, display_name, created_at)

messages                                     ── append-only, immutable once sent
   id              (UUID, globally unique)
   user_id         (partition key — owner's mailbox)
   thread_id       (groups messages into conversations)
   from_addr
   to_addrs[]
   cc[], bcc[]
   subject
   body_ref        (url into object store — NOT the full body)
   size_bytes
   received_at
   labels[]        ("inbox", "sent", "starred", custom user labels)
   is_read, is_spam
   message_id_hdr  (RFC 5322 "Message-ID:" header — used for threading)
   in_reply_to     (parent "Message-ID:" header)
   refs[]          (chain of ancestor Message-IDs)

threads
   id, user_id, subject_norm, last_msg_at, msg_count

attachments
   id, message_id, filename, content_type, size, storage_ref, sha256
```

A single real write (one inbound email) touches **3 systems**: metadata DB, object
store (body + attachments), and a new-message event for the search indexer.

## 2. Typed request/response models with Pydantic

Using **Pydantic** gives us free validation + JSON (de)serialization — perfect for
API boundaries.

In [1]:
from __future__ import annotations
from pydantic import BaseModel, EmailStr, Field
from datetime import datetime, timezone
from typing import Literal
import uuid

class SendEmailRequest(BaseModel):
    to: list[EmailStr] = Field(min_length=1)
    cc: list[EmailStr] = []
    bcc: list[EmailStr] = []
    subject: str = Field(max_length=998)     # RFC 5322 line-length limit
    body: str
    in_reply_to: str | None = None           # Message-ID of parent

class Message(BaseModel):
    id: str
    user_id: str
    thread_id: str
    from_addr: EmailStr
    to_addrs: list[EmailStr]
    subject: str
    body_ref: str                            # object-store key, e.g. "s3://mail/u123/msgs/abc.eml"
    received_at: datetime
    labels: list[str] = ["inbox"]
    is_read: bool = False

req = SendEmailRequest(
    to=["alice@example.com"],
    subject="Lunch?",
    body="Free at 12?",
)
print(req.model_dump_json(indent=2))

{
  "to": [
    "alice@example.com"
  ],
  "cc": [],
  "bcc": [],
  "subject": "Lunch?",
  "body": "Free at 12?",
  "in_reply_to": null
}


In [2]:
# What happens when validation fails? Try uncommenting the bad send.
from pydantic import ValidationError
try:
    SendEmailRequest(to=["not-an-email"], subject="hi", body="x")
except ValidationError as e:
    print("✔ rejected invalid address:")
    print(e.errors()[0]["msg"])

✔ rejected invalid address:
value is not a valid email address: An email address must have an @-sign.


## 3. Threading — bad → best

A "thread" is a conversation: reply, reply-to-reply, forward, etc. How do we
group messages into threads?

### v1 — normalize the subject line (BAD)

Strip `Re:`, `Fwd:`, whitespace. Group by the normalized subject.

**Breaks** when two unrelated people happen to use the same subject. Gmail literally
had this bug on cross-thread mis-grouping for short subjects like "hi".

In [3]:
import re

def normalize_subject(s: str) -> str:
    # strip leading Re:/Fwd: (any number), collapse whitespace, lowercase
    s = re.sub(r"^(\s*(re|fw|fwd):\s*)+", "", s, flags=re.I)
    return re.sub(r"\s+", " ", s).strip().lower()

samples = ["Re: Re:   Lunch?", "Fwd: Lunch?", "Lunch?", "  Hi  "]
for s in samples:
    print(f"{s!r:25} → {normalize_subject(s)!r}")

'Re: Re:   Lunch?'        → 'lunch?'
'Fwd: Lunch?'             → 'lunch?'
'Lunch?'                  → 'lunch?'
'  Hi  '                  → 'hi'


### v2 — RFC 5322 headers (BETTER)

Real email clients include three headers that form a tree:

- `Message-ID: <xyz@host>` — unique ID of this message.
- `In-Reply-To: <xyz@host>` — direct parent.
- `References: <a@...> <b@...> <c@...>` — the chain of ancestors, oldest first.

So the **thread ID is the Message-ID of the first message in the chain.**
If a reply arrives with `In-Reply-To: <old-msg@...>`, we look up that message, take
its `thread_id`, and use the same one.

In [4]:
# Minimal threading builder that uses headers but falls back to subject.
from dataclasses import dataclass, field

@dataclass
class Msg:
    msg_id: str                    # "<abc@host>"
    subject: str
    in_reply_to: str | None = None
    refs: list[str] = field(default_factory=list)
    thread_id: str | None = None

class ThreadStore:
    def __init__(self):
        self.by_msg_id: dict[str, Msg] = {}
        self.by_subject: dict[str, str] = {}   # normalized-subj → thread_id

    def add(self, m: Msg) -> str:
        # 1) try headers
        parent_id = m.in_reply_to or (m.refs[-1] if m.refs else None)
        parent = self.by_msg_id.get(parent_id) if parent_id else None
        if parent:
            m.thread_id = parent.thread_id
        else:
            # 2) subject fallback — only if we've seen it very recently
            key = normalize_subject(m.subject)
            m.thread_id = self.by_subject.get(key) or m.msg_id
            self.by_subject[key] = m.thread_id
        self.by_msg_id[m.msg_id] = m
        return m.thread_id

ts = ThreadStore()
a = Msg("<1@x>", "Project kickoff")
b = Msg("<2@x>", "Re: Project kickoff", in_reply_to="<1@x>", refs=["<1@x>"])
c = Msg("<3@x>", "Re: Project kickoff", in_reply_to="<2@x>", refs=["<1@x>", "<2@x>"])
d = Msg("<4@x>", "Project kickoff")   # different sender, no headers — risky

for m in [a, b, c, d]:
    print(f"{m.msg_id}  thread={ts.add(m)}")

<1@x>  thread=<1@x>
<2@x>  thread=<1@x>
<3@x>  thread=<1@x>
<4@x>  thread=<1@x>


Notice how `<4@x>` — a message with no threading headers — gets grouped into the
same thread as `<1@x>` by the subject fallback. In production you'd also require
the **same participant set** before falling back to subject, to prevent strangers
from silently joining a thread.

## 4. Core HTTP API

```http
GET  /messages?label=inbox&limit=50&cursor=...       # list
GET  /messages/{id}                                  # full message (incl. body ref)
POST /messages                                       # send
POST /messages/{id}/labels { add:[...], remove:[...] }
POST /messages/{id}/read                             # mark read
GET  /search?q=from:alice subject:report after:2025/01/01
GET  /attachments/{id}                               # pre-signed URL redirect
```

### Pagination — bad vs best

- **BAD:** `?page=42&size=50` → offsets are O(N) in most DBs and double-show or skip
  messages when new mail arrives.
- **BEST:** **cursor pagination** — the cursor is an opaque token encoding the
  `(received_at, id)` of the last row seen. O(1), stable under inserts.

In [5]:
# Cursor pagination demo
import base64, json
from datetime import datetime, timedelta, timezone

# simulated inbox sorted by received_at DESC
inbox = [
    {"id": f"m{i}", "received_at": datetime.now(timezone.utc) - timedelta(minutes=i), "subject": f"hi {i}"}
    for i in range(12)
]

def encode(cursor: dict) -> str:
    return base64.urlsafe_b64encode(json.dumps(cursor, default=str).encode()).decode()

def decode(token: str) -> dict:
    return json.loads(base64.urlsafe_b64decode(token.encode()).decode())

def list_inbox(limit=5, cursor_token: str | None = None):
    start = 0
    if cursor_token:
        c = decode(cursor_token)
        for i, m in enumerate(inbox):
            if m["id"] == c["last_id"]:
                start = i + 1
                break
    page = inbox[start:start + limit]
    next_token = encode({"last_id": page[-1]["id"]}) if len(page) == limit else None
    return page, next_token

page, tok = list_inbox(limit=5)
print("page 1:", [m["id"] for m in page])
page, tok = list_inbox(limit=5, cursor_token=tok)
print("page 2:", [m["id"] for m in page])
page, tok = list_inbox(limit=5, cursor_token=tok)
print("page 3:", [m["id"] for m in page], "next=", tok)

page 1: ['m0', 'm1', 'm2', 'm3', 'm4']
page 2: ['m5', 'm6', 'm7', 'm8', 'm9']
page 3: ['m10', 'm11'] next= None


## 5. Parsing incoming MIME with the stdlib

Real email is **MIME** — multipart containers with nested parts for text, HTML, and
attachments. Python's stdlib `email` module parses it without any extra dependency.

In [6]:
from email import message_from_string
from email.policy import default

raw = '''From: alice@ex.com
To: bob@us.com
Subject: Budget
Message-ID: <abc@ex.com>
In-Reply-To: <old@ex.com>
References: <old@ex.com>
MIME-Version: 1.0
Content-Type: multipart/mixed; boundary=BOUND

--BOUND
Content-Type: text/plain

Here's the budget.

--BOUND
Content-Type: text/csv; name="q3.csv"
Content-Disposition: attachment; filename="q3.csv"

item,amount
servers,10000
--BOUND--
'''

msg = message_from_string(raw, policy=default)
print("from:        ", msg["From"])
print("subject:     ", msg["Subject"])
print("message-id:  ", msg["Message-ID"])
print("in-reply-to: ", msg["In-Reply-To"])
print("--- parts ---")
for part in msg.walk():
    if part.is_multipart():
        continue
    disp = part.get("Content-Disposition", "inline").split(";")[0]
    print(f"  {part.get_content_type():15} {disp:10} {part.get_filename() or ''}")

from:         alice@ex.com
subject:      Budget
message-id:   <abc@ex.com>
in-reply-to:  <old@ex.com>
--- parts ---
  text/plain      inline     
  text/csv        attachment q3.csv


## 6. What about send?

`POST /messages` is deceptively tricky. The naive path would be to open a TCP
connection to each recipient's MX host synchronously — but that can take seconds
and sometimes fails. Real systems:

1. **Validate + render** the message, assign a `Message-ID`.
2. **Store** in the sender's `Sent` folder immediately (in our DB + object store).
3. **Enqueue** an outbound-delivery job per recipient domain.
4. A relay worker reads the queue, does SMTP delivery, retries with **exponential
   backoff**, and on final failure writes a **bounce** email back to the sender.

We'll simulate retries in notebook 3.

### Idempotency
Clients often retry `POST /messages` after a network blip. Require an
`Idempotency-Key` header; the server stores a short-lived mapping of
`key → message_id` so duplicates don't create duplicate sends.